# Milestone 1 — Preprocessing

End-to-end pipeline producing `data/processed/patients_clean.csv` and persisting fitted
imputer-state + scaler for downstream milestones.


In [1]:
import sys; sys.path.append('..')
import joblib
import json
import pandas as pd
from pathlib import Path

from src.data_loader import load_ckd
from src.preprocessing import (
    clean_types, impute_clinical, flag_outliers_iqr, encode_binary,
    fit_scale_numeric, NUMERIC_COLS, BINARY_COLS,
)

DATA = Path('../data'); MODELS = Path('../models')
DATA.joinpath('processed').mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)


## 1. Load + clean types

In [2]:
df = clean_types(load_ckd('../data/raw/kidney_disease.csv'))
print(df.shape)
df.dtypes


(400, 25)


age               float64
bp                float64
sg                float64
al                float64
su                float64
rbc                object
pc                 object
pcc                object
ba                 object
bgr               float64
bu                float64
sc                float64
sod               float64
pot               float64
hemo              float64
pcv               float64
wc                float64
rc                float64
htn                object
dm                 object
cad                object
appet              object
pe                 object
ane                object
classification     object
dtype: object

## 2. Group-aware imputation

In [3]:
df_imp, fitted_imputer = impute_clinical(df)
assert df_imp.isna().sum().sum() == 0, 'still has NaNs!'
df_imp.to_csv('../data/interim/patients_imputed.csv', index=False)
print('Imputation complete; 0 missing values remaining.')


Imputation complete; 0 missing values remaining.


## 3. Outlier flagging (no removal)

In [4]:
df_flag = flag_outliers_iqr(df_imp, cols=list(NUMERIC_COLS))
flag_cols = [c for c in df_flag.columns if c.endswith('_outlier_flag')]
print('Outlier counts per feature:')
print(df_flag[flag_cols].sum().sort_values(ascending=False))


Outlier counts per feature:
su_outlier_flag      61
sc_outlier_flag      51
bu_outlier_flag      38
bp_outlier_flag      36
bgr_outlier_flag     36
sod_outlier_flag     18
wc_outlier_flag      17
pot_outlier_flag     14
age_outlier_flag     10
sg_outlier_flag       7
hemo_outlier_flag     1
pcv_outlier_flag      1
rc_outlier_flag       1
al_outlier_flag       0
dtype: int64


## 4. Binary encoding

In [5]:
df_enc = encode_binary(df_flag)
df_enc.head()


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,su_outlier_flag,bgr_outlier_flag,bu_outlier_flag,sc_outlier_flag,sod_outlier_flag,pot_outlier_flag,hemo_outlier_flag,pcv_outlier_flag,wc_outlier_flag,rc_outlier_flag
0,48.0,80.0,1.020,1.0,0.0,0,0,0,0,121.0,...,0,0,0,0,0,0,0,0,0,0
1,7.0,50.0,1.020,4.0,0.0,0,0,0,0,111.0,...,0,0,0,0,0,0,0,0,0,0
2,62.0,80.0,1.010,2.0,3.0,0,0,0,0,423.0,...,1,1,0,0,0,0,0,0,0,0
3,48.0,70.0,1.005,4.0,0.0,0,1,1,0,117.0,...,0,0,0,0,1,1,0,0,0,0
4,51.0,80.0,1.010,2.0,0.0,0,0,0,0,106.0,...,0,0,0,0,0,0,0,0,0,0


## 5. Scaling

In [6]:
df_scaled, scaler = fit_scale_numeric(df_enc, cols=list(NUMERIC_COLS))
print('Numeric means after scaling (should be ~0):')
print(df_scaled[list(NUMERIC_COLS)].mean().round(6))


Numeric means after scaling (should be ~0):
age     0.0
bp     -0.0
sg      0.0
al     -0.0
su     -0.0
bgr     0.0
bu      0.0
sc     -0.0
sod    -0.0
pot    -0.0
hemo   -0.0
pcv     0.0
wc      0.0
rc      0.0
dtype: float64


## 6. Persist artifacts

In [7]:
# Save cleaned dataset
df_scaled.to_csv('../data/processed/patients_clean.csv', index=False)

# Persist scaler
joblib.dump(scaler, '../models/scaler.pkl')

# Persist imputer (callable; used by dashboard for new-patient prediction)
from src.preprocessing import ClinicalImputer
imputer = ClinicalImputer().fit(df)  # fit on the post-clean_types frame
joblib.dump(imputer, '../models/imputer.pkl')

# Also persist raw fitted state for inspectability
with open('../models/imputer_state.json', 'w') as f:
    json.dump(fitted_imputer, f, indent=2, default=str)

print('Saved:')
print('  data/processed/patients_clean.csv:', df_scaled.shape)
print('  models/scaler.pkl')
print('  models/imputer.pkl')
print('  models/imputer_state.json')


Saved:
  data/processed/patients_clean.csv: (400, 39)
  models/scaler.pkl
  models/imputer.pkl
  models/imputer_state.json


## 7. Sanity checks
- Row count preserved (no rows dropped).
- Zero NaNs.
- Numeric columns z-score scaled.
- Binary flag columns remain 0/1.
- Outlier flag columns remain 0/1.
- Held-out target `classification` preserved as 0/1.


In [8]:
print('Rows:', len(df_scaled), '== raw rows:', len(df))
print('NaNs:', df_scaled.isna().sum().sum())
print('classification unique:', sorted(df_scaled['classification'].unique()))


Rows: 400 == raw rows: 400
NaNs: 0
classification unique: [0, 1]
